# 06 – Data Quality Checks

Runs configurable DQ checks on silver tables:
- **Not null:** Validates required fields are populated
- **Range:** Validates numeric values within expected bounds
- **Uniqueness:** Validates primary key uniqueness
- **Referential integrity:** Validates foreign key relationships

**Source:** `silver.{books, users, ratings}`
**Target:** `gold.dq_report` (check results table)
**Config:** `config/06_dq_checks.json`

In [0]:
import json, sys, importlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc/config/00_setup.json"
with open(SETUP_CONFIG) as f:
    setup = json.load(f)

sys.path.insert(0, setup["project_path"])

for mod in [m for m in list(sys.modules.keys()) if m.startswith("source")]:
    del sys.modules[mod]

DQChecks = importlib.import_module("source.06_dq_checks").DQChecks

CONFIG_PATH = f"{setup['project_path']}/config/06_dq_checks.json"

In [0]:
pipeline = DQChecks(config_path=CONFIG_PATH)
report_df = pipeline.run()

In [0]:
display(report_df.orderBy("status", "table_name"))

In [0]:
from pyspark.sql import functions as F

failed = report_df.filter(F.col("status") == "FAIL")
print(f"Total checks: {report_df.count()}")
print(f"Passed: {report_df.filter(F.col('status') == 'PASS').count()}")
print(f"Failed: {failed.count()}")

if failed.count() > 0:
    print("\n=== Failed Checks ===")
    display(failed)